# Tema 1.3 - CompraFácil
## Práctica guiada: evaluación del impacto de las soluciones de ciencia de datos

Este notebook está homologado con el estilo de los cuadernos anteriores y organizado por **pasos** para facilitar su uso en Google Colab.

### Propósito de la práctica
En esta práctica aplicarás un diseño experimental **A/B** para evaluar si una solución de ciencia de datos genera un impacto real y medible en el negocio.

Trabajarás con el caso **CompraFácil**, donde un modelo predictivo permitió identificar clientes con alta probabilidad de abandonar su carrito. A partir de ello, se implementó una estrategia experimental con dos grupos:

- **Grupo Control**: clientes que no recibieron recordatorios.
- **Grupo Tratamiento**: clientes que recibieron recordatorios personalizados generados por el modelo.

### Objetivos de análisis
Determinar:
1. si la intervención aumentó la tasa de conversión,
2. si la diferencia observada es estadísticamente significativa,
3. si la intervención genera un beneficio financiero justificable,
4. y qué implicaciones éticas deben considerarse antes de escalar la estrategia.

### Archivos de salida que generará este notebook
- `ab_group_conversion.csv`
- `ab_test_summary.csv`
- `ab_financial_impact.csv`
- `ab_executive_summary.csv`


In [ ]:
# ============================================================
# 0. Configuración inicial
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown
from statsmodels.stats.proportion import proportions_ztest

SEED = 42
rng = np.random.default_rng(SEED)

display(Markdown("### Librerías cargadas correctamente"))
print("Entorno listo para la práctica 3 de CompraFácil.")


## Paso 1. Importar librerías y generar datos

En esta primera sección se simulan los datos del experimento A/B.  
Cada fila representa un cliente y la variable `conversion` indica si completó o no la compra.

En el diseño experimental:
- el grupo **Control** no recibe recordatorios,
- el grupo **Tratamiento** recibe recordatorios personalizados basados en el modelo predictivo.

La intención es contrastar ambas condiciones para determinar si la intervención realmente modifica el comportamiento del usuario.


In [ ]:
# ============================================================
# 1. Generación de datos simulados del experimento A/B
# ============================================================

n_control = 1000
n_treatment = 1000

p_control = 0.184
p_treatment = 0.224

control_df = pd.DataFrame({
    "customer_id": np.arange(1, n_control + 1),
    "group": "Control",
    "conversion": rng.binomial(1, p_control, size=n_control)
})

treatment_df = pd.DataFrame({
    "customer_id": np.arange(n_control + 1, n_control + n_treatment + 1),
    "group": "Treatment",
    "conversion": rng.binomial(1, p_treatment, size=n_treatment)
})

ab_df = pd.concat([control_df, treatment_df], ignore_index=True)

display(Markdown("### Vista inicial del experimento"))
display(ab_df.head())

display(Markdown("### Tamaño del experimento"))
print(ab_df["group"].value_counts())


## Paso 2. Calcular las tasas de conversión

A continuación se calcula la tasa de conversión promedio para cada grupo.

Esta es la primera evidencia descriptiva del posible impacto de la intervención.  
Sin embargo, una diferencia observada entre grupos no basta por sí sola para concluir que el efecto es real: todavía será necesario evaluar su significancia estadística.


In [ ]:
# ============================================================
# 2. Tasas de conversión por grupo
# ============================================================

conversion_summary = (
    ab_df.groupby("group")
    .agg(
        total_customers=("conversion", "count"),
        conversions=("conversion", "sum"),
        conversion_rate=("conversion", "mean")
    )
    .reset_index()
)

display(Markdown("### Tasa de conversión por grupo"))
display(conversion_summary)

control_rate = conversion_summary.loc[conversion_summary["group"] == "Control", "conversion_rate"].iloc[0]
treatment_rate = conversion_summary.loc[conversion_summary["group"] == "Treatment", "conversion_rate"].iloc[0]
uplift = treatment_rate - control_rate

print(f"Conversión grupo Control: {control_rate:.4f}")
print(f"Conversión grupo Tratamiento: {treatment_rate:.4f}")
print(f"Uplift absoluto observado: {uplift:.4f}")


## Paso 3. Prueba de significancia estadística

En este paso aplicarás una **prueba z de proporciones** para contrastar si la diferencia entre ambas tasas de conversión puede considerarse estadísticamente significativa.

### Regla de interpretación
- Si `p < 0.05`, existe evidencia suficiente para afirmar que la intervención produjo un cambio significativo.
- Si `p >= 0.05`, la diferencia observada podría deberse al azar.


In [ ]:
# ============================================================
# 3. Prueba z para comparación de proporciones
# ============================================================

successes = conversion_summary["conversions"].values
observations = conversion_summary["total_customers"].values

z_stat, p_value = proportions_ztest(count=successes, nobs=observations, alternative="two-sided")

significance_df = pd.DataFrame([{
    "z_statistic": round(float(z_stat), 4),
    "p_value": round(float(p_value), 6),
    "is_significant_05": bool(p_value < 0.05),
    "control_rate": round(float(control_rate), 4),
    "treatment_rate": round(float(treatment_rate), 4),
    "uplift_absolute": round(float(uplift), 4),
    "uplift_relative_pct": round(float((uplift / control_rate) * 100), 2) if control_rate > 0 else np.nan
}])

display(Markdown("### Resultado de la prueba de significancia"))
display(significance_df)

if p_value < 0.05:
    print("Interpretación: la diferencia entre grupos es estadísticamente significativa al 5%.")
else:
    print("Interpretación: no existe evidencia suficiente para afirmar que la diferencia sea significativa al 5%.")


## Paso 4. Visualizar la diferencia de conversión

Una visualización ayuda a comunicar el resultado del experimento de forma más directa.

En este caso se utiliza una gráfica de barras para comparar la tasa de conversión entre el grupo de Control y el grupo de Tratamiento.

Reflexiona mientras observas la gráfica:
- ¿La diferencia visual coincide con el resultado del test estadístico?
- ¿Este gráfico sería suficiente para convencer a un directivo?


In [ ]:
# ============================================================
# 4. Gráfica comparativa de conversión
# ============================================================

plot_df = conversion_summary.copy()
plot_df["conversion_rate_pct"] = plot_df["conversion_rate"] * 100

plt.figure(figsize=(8, 5))
plt.bar(plot_df["group"], plot_df["conversion_rate_pct"])
plt.title("Tasa de conversión por grupo")
plt.xlabel("Grupo")
plt.ylabel("Conversión (%)")
plt.ylim(0, max(plot_df["conversion_rate_pct"]) * 1.25)
for idx, row in plot_df.iterrows():
    plt.text(idx, row["conversion_rate_pct"] + 0.5, f"{row['conversion_rate_pct']:.2f}%", ha="center")
plt.grid(axis="y", alpha=0.3)
plt.show()


## Paso 5. Estimar el ROI y el beneficio neto

Ahora traducirás el resultado del experimento a términos financieros.

Para ello se definen algunos parámetros económicos:
- ingreso promedio por conversión ganada,
- costo de enviar la intervención,
- y número adicional de conversiones atribuibles al tratamiento.

### Interpretación clave
- Un **ROI positivo** implica que la intervención es rentable.
- Un **ROI negativo** indica que el costo de la estrategia supera el beneficio esperado.


In [ ]:
# ============================================================
# 5. Cálculo del impacto financiero
# ============================================================

business_params = {
    "revenue_per_conversion": 180.0,
    "cost_per_intervention": 12.0
}

incremental_conversions = (treatment_rate - control_rate) * n_treatment
incremental_revenue = incremental_conversions * business_params["revenue_per_conversion"]
campaign_cost = n_treatment * business_params["cost_per_intervention"]
net_benefit = incremental_revenue - campaign_cost
roi_pct = (net_benefit / campaign_cost) * 100 if campaign_cost > 0 else 0.0

financial_impact_df = pd.DataFrame([{
    "control_rate": round(float(control_rate), 4),
    "treatment_rate": round(float(treatment_rate), 4),
    "uplift_absolute": round(float(uplift), 4),
    "uplift_relative_pct": round(float((uplift / control_rate) * 100), 2) if control_rate > 0 else np.nan,
    "incremental_conversions_est": round(float(incremental_conversions), 2),
    "revenue_per_conversion": business_params["revenue_per_conversion"],
    "incremental_revenue": round(float(incremental_revenue), 2),
    "campaign_cost": round(float(campaign_cost), 2),
    "net_benefit": round(float(net_benefit), 2),
    "roi_pct": round(float(roi_pct), 2)
}])

display(Markdown("### Impacto financiero estimado"))
display(financial_impact_df)

if roi_pct > 0:
    print("Interpretación financiera: la intervención es rentable.")
else:
    print("Interpretación financiera: la intervención no es rentable en su estado actual.")


## Paso 6. Crear un resumen ejecutivo

En este paso se construye una tabla con los indicadores clave para comunicar resultados a perfiles no técnicos.

La intención es producir una salida sintética que permita responder, de forma ejecutiva:
- si hubo mejora en conversión,
- si la mejora es significativa,
- y si la campaña justifica su costo.


In [ ]:
# ============================================================
# 6. Resumen ejecutivo
# ============================================================

if p_value < 0.05 and roi_pct > 0:
    executive_conclusion = (
        "La intervención presenta una mejora estadísticamente significativa en conversión "
        "y además genera un ROI positivo. Bajo estos resultados, la estrategia es candidata "
        "a escalarse de manera controlada."
    )
elif p_value < 0.05 and roi_pct <= 0:
    executive_conclusion = (
        "Aunque la intervención muestra una mejora estadísticamente significativa, "
        "el ROI es negativo. Antes de escalar la estrategia se requiere reducir costos "
        "o incrementar el valor económico por conversión."
    )
elif p_value >= 0.05 and roi_pct > 0:
    executive_conclusion = (
        "Se observa un ROI positivo, pero la diferencia entre grupos no es estadísticamente significativa. "
        "Se recomienda ampliar el experimento antes de tomar una decisión de despliegue."
    )
else:
    executive_conclusion = (
        "No existe evidencia estadística suficiente para afirmar que la intervención haya generado "
        "un impacto real, y además el ROI no justifica su implementación en el estado actual. "
        "Se recomienda optimizar la segmentación, reducir costos y revisar la estrategia."
    )

executive_summary_df = pd.DataFrame([{
    "control_conversion_rate": round(float(control_rate), 4),
    "treatment_conversion_rate": round(float(treatment_rate), 4),
    "uplift_absolute": round(float(uplift), 4),
    "z_statistic": round(float(z_stat), 4),
    "p_value": round(float(p_value), 6),
    "is_significant_05": bool(p_value < 0.05),
    "incremental_revenue": round(float(incremental_revenue), 2),
    "campaign_cost": round(float(campaign_cost), 2),
    "net_benefit": round(float(net_benefit), 2),
    "roi_pct": round(float(roi_pct), 2),
    "executive_conclusion": executive_conclusion
}])

display(Markdown("### Resumen ejecutivo"))
display(executive_summary_df)


## Paso 7. Reflexión ética y sostenibilidad

Más allá de la significancia estadística y del ROI, la evaluación de una solución analítica debe incorporar una revisión ética y de sostenibilidad.

Reflexiona sobre las siguientes preguntas:
- ¿Se obtuvo el consentimiento informado del usuario para usar sus datos?
- ¿El modelo podría afectar de forma diferenciada a ciertos segmentos?
- ¿Qué riesgos existirían si esta decisión se automatiza a gran escala?
- ¿La campaña podría derivar en prácticas invasivas o discriminatorias?

La adopción responsable de soluciones de ciencia de datos requiere equilibrio entre:
- valor económico,
- rigor técnico,
- respeto por los usuarios,
- y sostenibilidad a largo plazo.


In [ ]:
# ============================================================
# 7. Exportación de resultados
# ============================================================

conversion_summary.to_csv("ab_group_conversion.csv", index=False)
significance_df.to_csv("ab_test_summary.csv", index=False)
financial_impact_df.to_csv("ab_financial_impact.csv", index=False)
executive_summary_df.to_csv("ab_executive_summary.csv", index=False)

display(Markdown("### Archivos generados"))
print("- ab_group_conversion.csv")
print("- ab_test_summary.csv")
print("- ab_financial_impact.csv")
print("- ab_executive_summary.csv")

try:
    from google.colab import files
    files.download("ab_group_conversion.csv")
    files.download("ab_test_summary.csv")
    files.download("ab_financial_impact.csv")
    files.download("ab_executive_summary.csv")
except Exception:
    print("Si no estás en Google Colab, ignora este mensaje. Los archivos ya fueron creados en el directorio de trabajo.")


## Resumen final de la práctica

Este cuaderno:
1. simula un experimento A/B para el caso CompraFácil,
2. calcula tasas de conversión para Control y Tratamiento,
3. aplica una prueba z de proporciones,
4. visualiza la diferencia entre grupos,
5. estima beneficio neto y ROI,
6. genera un resumen ejecutivo,
7. y cierra con una reflexión ética sobre la sostenibilidad de la intervención.

Con ello, la práctica conecta evaluación estadística, interpretación de negocio y responsabilidad en la toma de decisiones.
